In [1]:
import numpy as np
from scipy.stats import poisson
import sys
sys.path.append('../../pysimARG')
from add_mutation_truncated import truncated_poisson
from clonal_genealogy import ClonalTree
from ClonalOrigin_pair import ARG
from localtree import LocalTree
from add_mutation_truncated import add_mutation_truncated

In [2]:
np.random.seed(100)
tree = ClonalTree(10)

rho_site = 0.5
theta_site = 0.3
L = 100
delta = 5
k = 10

In [3]:
arg = ARG(tree, rho_site, L, delta, k)

In [5]:
node_site = add_mutation_truncated(arg, theta_site)

ValueError: object of too small depth for desired array

In [6]:
num_site = arg.node_mat.shape[1]
n_list = np.zeros(num_site, dtype=int)
mutate_edge = list()
mutate_site = list()

# Initialize node_site matrix (boolean)
node_site = np.zeros(
    (arg.node_mat.shape[0], arg.node_mat.shape[1]), dtype=bool
)

# Simulate mutations for each site
for i in range(num_site):
    # Length of local tree without reduction
    local_edge = np.where(arg.edge_mat[:, i])[0]
    local_length = arg.edge[local_edge, 2]
    # Truncated Poisson distribution
    local_n = truncated_poisson(theta_site * np.sum(local_length) / 2)
    n_list[i] = local_n
    mutate_site.extend([i] * local_n)
    # Simulate edges
    probs = local_length / np.sum(local_length)
    mutate_edge.extend(np.random.choice(local_edge, local_n, replace=True, p=probs).tolist())
    
n_list, mutate_edge, mutate_site

(array([1, 1]), [44, 30], [0, 1])

In [ ]:
# Simulate the mutations at every node
# Process edges from last to first (bottom-up in the tree)
for i in range(len(arg.edge) - 1, -1, -1):
    print(i)
    edge_mutation = mutate_site[mutate_edge == i]

    parent_idx = int(arg.edge[i, 0]) - 1
    child_idx = int(arg.edge[i, 1]) - 1

    parent_seq = node_site[parent_idx, :].copy()
    flip_counts = np.bincount(edge_mutation, minlength=len(parent_seq))
    parent_seq = np.logical_xor(parent_seq, flip_counts % 2 == 1)

    material_range = arg.edge_mat[i, :]
    node_site[child_idx, material_range==True] = parent_seq[material_range==True]

ValueError: object of too small depth for desired array